# Predicción temprana de la desnutrición crónica infantil a partir del carné de control

Análisis con los microdatos de la Encuesta Nacional sobre Desnutrición Infantil (ENDI), Ronda 2 (2023-2024),
del Instituto Nacional de Estadística y Censos (INEC) de Ecuador. El notebook:

1. reconstruye las mediciones de talla transcritas del carné de control del niño sano y calcula su puntaje *Z*
   de talla para la edad con el patrón de crecimiento de la OMS;
2. compara el carné con la medición estandarizada de la encuesta: sesgo sistemático, error aleatorio,
   análisis de Bland-Altman y efecto del intervalo de tiempo entre ambas mediciones;
3. construye variables de trayectoria con las mediciones hasta los 18 meses, corregidas por el sesgo del carné;
4. compara cuatro conjuntos de predictores y tres algoritmos para predecir la desnutrición crónica infantil (DCI)
   entre los 24 y 42 meses, con validación cruzada agrupada por unidad primaria de muestreo (UPM);
5. evalúa la calibración y la clasificación del modelo final frente a reglas de corte aplicadas al carné;
6. entrena el modelo final y exporta sus parámetros para el módulo de inferencia `riesgo_dci.py`.

**Datos** (carpeta `datos/` o la ruta de la variable de entorno `ENDI_DATOS`; ver `datos/README.md`):

- `BDD_ENDI_R2_f1_personas.dta` y `BDD_ENDI_R2_f2_salud_ninez.dta`, del catálogo ANDA del INEC;
- `lms_talla_edad_oms.csv`, con los parámetros LMS de longitud/talla para la edad del patrón OMS.

**Salidas**: `resultados/tablas/` (CSV), `resultados/figuras/` (PNG, 300 dpi), `resultados/datos/` (predicciones y
pares carné-encuesta, sin identificadores), `resultados/cifras.json` y `modelo/parametros_modelo.json`.

**Convenciones**

- Las llaves de persona (`id_upm`, `id_viv`, `id_hogar`, `id_per`) se tratan como texto y todas las uniones usan `validate=`.
- Se excluyen los puntajes *Z* con |*Z*| > 6 en ambas fuentes (criterio OMS).
- El diseño muestral se usa para las prevalencias (factor de expansión) y para la validación (agrupación por UPM);
  el sesgo, la sensibilidad y el AUC se calculan sin ponderar.
- Todas las métricas de desempeño provienen de predicciones de validación cruzada agrupada, nunca del ajuste sobre
  la misma muestra.

**Reproducibilidad.** La asignación de conglomerados a pliegues de `GroupKFold` cambió entre versiones de
scikit-learn. Los valores de referencia de la sección 13 se obtuvieron con las versiones de `requirements.txt`
(scikit-learn 1.9.0); con otra versión, las métricas de validación cruzada pueden diferir en el tercer decimal.

## 1. Configuración

In [ ]:
import json
import os
import sys
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat
import scipy
import sklearn
from scipy import stats
from scipy.optimize import brentq
from scipy.special import expit, logit
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# raíz del repositorio: el notebook puede ejecutarse desde la raíz o desde notebooks/
REPO = Path.cwd() if (Path.cwd() / 'riesgo_dci.py').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))
import figuras
import riesgo_dci as rd

DATOS = Path(os.environ.get('ENDI_DATOS', REPO / 'datos'))
RESULTADOS = REPO / 'resultados'
MODELO = REPO / 'modelo'
for carpeta in (RESULTADOS / 'tablas', RESULTADOS / 'figuras', RESULTADOS / 'datos', MODELO):
    carpeta.mkdir(parents=True, exist_ok=True)

F = lambda nombre: str(DATOS / f'BDD_ENDI_R2_{nombre}.dta')
K = ['id_upm', 'id_viv', 'id_hogar', 'id_per']      # llave de la persona

UMBRAL = 0.20            # umbral de clasificación del modelo final (prioriza la sensibilidad)
N_PLIEGUES = 5           # validación cruzada agrupada por UPM
N_BOOT = 2000            # réplicas del bootstrap de conglomerados
SEMILLA_BOOT = 2026
SEMILLA_MODELOS = 42

CIFRAS = {}              # cifras calculadas; se guardan en resultados/cifras.json
pd.set_option('display.width', 140)

VERSIONES = {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__,
             'scikit-learn': sklearn.__version__, 'scipy': scipy.__version__, 'pyreadstat': pyreadstat.__version__}
print(' | '.join(f'{k} {v}' for k, v in VERSIONES.items()))
if sklearn.__version__ != '1.9.0':
    warnings.warn('Los valores de referencia se obtuvieron con scikit-learn 1.9.0; con otra versión la asignación de '
                  'pliegues de GroupKFold puede cambiar y las métricas de validación cruzada diferir levemente.')

In [ ]:
def rango(s, lo, hi):
    '''Convierte a número y anula los valores fuera de [lo, hi], incluidos los códigos de no respuesta (8888, 9999).'''
    x = pd.to_numeric(s, errors='coerce')
    return x.where((x >= lo) & (x <= hi))


def cobertura(df, cols):
    '''Proporción de filas con dato válido; en las llaves de texto, '' cuenta como ausente.'''
    return pd.Series({c: (df[c].notna() & (df[c].astype(str) != '')).mean() for c in cols}).sort_values().round(3)


def n_ninos(df):
    '''Número de niños distintos en una tabla de mediciones.'''
    return int(df.groupby(K).ngroups) if len(df) else 0


def sens_espec(prueba_positiva, referencia_positiva):
    '''Sensibilidad y especificidad de una prueba binaria frente a una referencia binaria.'''
    t, r = np.asarray(prueba_positiva, bool), np.asarray(referencia_positiva, bool)
    return (t & r).sum() / r.sum(), (~t & ~r).sum() / (~r).sum()


def operativas(y, marca):
    '''Matriz de confusión y métricas de una regla de clasificación.'''
    y = np.asarray(y).astype(int)
    marca = np.asarray(marca).astype(bool)
    vp = int((marca & (y == 1)).sum()); fp = int((marca & (y == 0)).sum())
    fn = int((~marca & (y == 1)).sum()); vn = int((~marca & (y == 0)).sum())
    return dict(marcados=marca.mean(), sens=vp / (vp + fn), espec=vn / (vn + fp),
                vpp=vp / (vp + fp) if (vp + fp) else np.nan, vp=vp, fp=fp, fn=fn, vn=vn)


def pipeline_logistico(num, cat=()):
    '''Regresión logística: imputación por mediana y estandarización de las variables numéricas,
    codificación one-hot de las categóricas.'''
    pasos = []
    if num:
        pasos.append(('n', Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler())]), list(num)))
    if cat:
        pasos.append(('c', OneHotEncoder(handle_unknown='ignore', drop='first'), list(cat)))
    return Pipeline([('prep', ColumnTransformer(pasos)), ('clf', LogisticRegression(max_iter=2000))])


def mco(y, columnas, nombres):
    '''Mínimos cuadrados ordinarios con errores estándar clásicos; devuelve una tabla de coeficientes.'''
    X = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in columnas])
    yv = np.asarray(y, float)
    beta, *_ = np.linalg.lstsq(X, yv, rcond=None)
    res = yv - X @ beta
    gl = len(yv) - X.shape[1]
    ee = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * (res @ res) / gl)
    p = 2 * stats.t.sf(np.abs(beta / ee), gl)
    return pd.DataFrame({'coef': beta, 'ee': ee, 'p': p}, index=['intercepto'] + nombres)


def guardar_tabla(df, nombre):
    df.to_csv(RESULTADOS / 'tablas' / f'{nombre}.csv', index=False, encoding='utf-8')
    return df

## 2. Base de niños

Niños de 0 a 59 meses con estado nutricional válido en `f1_personas`, con su sexo, edad, variables
sociodemográficas, instrucción de la madre (vinculada mediante `f2_salud_ninez`) y tipo de nacimiento.

In [ ]:
cols_per = ['id_upm', 'id_viv', 'id_hogar', 'id_per', 'id_mef', 'fexp', 'estrato', 'prov', 'area', 'etnia',
            'edaddias', 'f1_s1_2', 'dcronica', 'dcronica2_5', 'nivins_mef']
per_todos, _ = pyreadstat.read_dta(F('f1_personas'), usecols=cols_per)

# instrucción de las madres: son otras filas del mismo hogar, por eso se extrae antes de filtrar a los niños
madres = (per_todos.loc[pd.to_numeric(per_todos.nivins_mef, errors='coerce').notna(),
                        ['id_upm', 'id_viv', 'id_hogar', 'id_mef', 'nivins_mef']]
          .rename(columns={'nivins_mef': 'educ_madre'}))

per = per_todos[per_todos.dcronica.notna()].copy()        # niños con estado nutricional (0 a 59 meses)
per['edad_m'] = pd.to_numeric(per.edaddias, errors='coerce') / 30.4375
per['sexo'] = pd.to_numeric(per.f1_s1_2, errors='coerce').map({1: 'M', 2: 'F'})

# el id_mef del niño está vacío en f1_personas; el vínculo con la madre está en f2_salud_ninez
puente, _ = pyreadstat.read_dta(F('f2_salud_ninez'), usecols=K + ['id_mef'])
nin = (per.drop(columns=['id_mef', 'nivins_mef'])
          .merge(puente, on=K, how='left', validate='1:1')
          .merge(madres, on=['id_upm', 'id_viv', 'id_hogar', 'id_mef'], how='left', validate='m:1'))

CIFRAS['flujo.n_ninos'] = len(nin)
print(f'niños con estado nutricional válido: {len(nin):,}')
print('\ncobertura de variables clave:')
print(cobertura(nin, ['educ_madre', 'etnia', 'area', 'sexo', 'dcronica2_5']).to_string())

In [ ]:
# peso al nacer, semanas de gestación y tipo de nacimiento
cols_nac = K + ['f2_s4d_432', 'f2_s4d_443_b', 'f2_s4d_444']
nac, _ = pyreadstat.read_dta(F('f2_salud_ninez'), usecols=cols_nac)

nac['peso_nacer'] = rango(nac.f2_s4d_443_b, 500, 6000).fillna(rango(nac.f2_s4d_444, 500, 6000))
nac['sem_gest'] = rango(nac.f2_s4d_432, 24, 45)
nac['prematuro'] = (nac.sem_gest < 37).astype(float).where(nac.sem_gest.notna())

# pequeño para la edad gestacional (PEG): por debajo del percentil 10 interno del peso por semana de gestación.
# Es una aproximación para comparar grupos dentro de la ENDI, no un estándar externo como INTERGROWTH-21st.
p10 = nac.groupby('sem_gest').peso_nacer.transform(lambda s: s.quantile(.10))
nac['peg'] = (nac.peso_nacer < p10).astype(float).where(nac.peso_nacer.notna())

nac['tipo_nac'] = np.select(
    [(nac.prematuro == 1) & (nac.peg == 1),
     (nac.prematuro == 1) & (nac.peg == 0),
     (nac.prematuro == 0) & (nac.peg == 1)],
    ['prematuro_peg', 'prematuro_aeg', 'termino_peg'], default='termino_aeg')
nac.loc[nac.peso_nacer.isna() | nac.sem_gest.isna(), 'tipo_nac'] = None

nin = nin.merge(nac[K + ['tipo_nac']], on=K, how='left', validate='1:1')
print('cobertura del tipo de nacimiento:', round(nin.tipo_nac.notna().mean(), 3))

## 3. Mediciones del carné de control y puntaje *Z*

El carné transcrito tiene hasta doce mediciones por niño (edad en meses, peso y talla). Se descartan valores
físicamente implausibles y se calcula el puntaje *Z* de talla para la edad con el método LMS de la OMS.

In [ ]:
piezas = ['edad', 'peso', 'talla']
cols_carne = K + [f'f2_s4f_464_{p}_{i}' for i in range(1, 13) for p in piezas]
tr, _ = pyreadstat.read_dta(F('f2_salud_ninez'), usecols=cols_carne)

largo = []
for i in range(1, 13):
    d = tr[K].copy()
    d['orden'] = i
    for p in piezas:
        d[p] = pd.to_numeric(tr[f'f2_s4f_464_{p}_{i}'], errors='coerce')
    largo.append(d)
largo = pd.concat(largo, ignore_index=True)
con_talla = largo.dropna(subset=['talla'])

L = largo.copy()
L['edad'] = L.edad.where(L.edad.between(0, 60))
L['peso'] = L.peso.where(L.peso.between(1.5, 30))
L['talla'] = L.talla.where(L.talla.between(*rd.RANGO_TALLA_CM))
L = L.dropna(subset=['edad', 'talla'])

print(f'mediciones con talla registrada: {len(con_talla):,} ({n_ninos(con_talla):,} niños)')
print(f'tras el filtro de rango físico:  {len(L):,}  (-{len(con_talla) - len(L):,})')

In [ ]:
# Parámetros LMS de la OMS. Si el mes 24 aparece dos veces (longitud en la tabla de 0 a 2 años y talla en la de
# 2 a 5 años), se conserva la última fila, que corresponde a la talla.
lms = (pd.read_csv(DATOS / 'lms_talla_edad_oms.csv')
         .sort_values(['sexo', 'mes'])
         .drop_duplicates(['sexo', 'mes'], keep='last')[['sexo', 'mes', 'L', 'M', 'S']])
assert len(lms) == 122 and set(lms.sexo) == {'M', 'F'}, \
    f'la tabla LMS debe tener 122 filas (sexos M y F, meses 0 a 60); tiene {len(lms)}'

L2 = L.merge(nin[K + ['sexo']], on=K, how='left', validate='m:1')
L2['mes'] = L2.edad.round().astype(int).clip(0, 60)
L2 = L2.merge(lms, on=['sexo', 'mes'], how='left', validate='m:1')
L2['z'] = rd.puntaje_z(L2.talla, L2.L, L2.M, L2.S)
L2['z_ok'] = L2.z.where(L2.z.between(-rd.Z_MAX, rd.Z_MAX))     # |Z| > 6: error de medición (OMS)

validas = L2[L2.z_ok.notna()]
FLUJO = dict(n_ninos=len(nin), n_carne=n_ninos(con_talla), med_carne=len(con_talla),
             exc_rango=len(con_talla) - len(L), exc_lms=len(L) - int(L2.z.notna().sum()),
             exc_z6=int(L2.z.notna().sum() - L2.z_ok.notna().sum()),
             med_validas=len(validas), n_validos=n_ninos(validas))
print(f"sin sexo o parámetros LMS: -{FLUJO['exc_lms']:,} | |Z| > 6: -{FLUJO['exc_z6']:,}")
print(f"mediciones válidas: {FLUJO['med_validas']:,} de {FLUJO['n_validos']:,} niños")

## 4. Medición estandarizada de la encuesta

Promedio de las tomas repetidas de longitud (en decúbito) o, si falta, de talla (de pie) del levantamiento
antropométrico de la ENDI. Es la referencia frente a la cual se evalúa el carné.

In [ ]:
cols_ant = K + ['f1_s5_5_1', 'f1_s5_5_2', 'f1_s5_5_3',     # longitud (decúbito)
                'f1_s5_6_1', 'f1_s5_6_2', 'f1_s5_6_3']    # talla (de pie)
ant, _ = pyreadstat.read_dta(F('f1_personas'), usecols=cols_ant)
for c in cols_ant[len(K):]:
    ant[c] = rango(ant[c], *rd.RANGO_TALLA_CM)

ant['medida'] = (ant[['f1_s5_5_1', 'f1_s5_5_2', 'f1_s5_5_3']].mean(axis=1)
                 .fillna(ant[['f1_s5_6_1', 'f1_s5_6_2', 'f1_s5_6_3']].mean(axis=1)))
ant = ant.merge(nin[K + ['sexo', 'edad_m']], on=K, how='inner', validate='1:1')
ant['mes'] = ant.edad_m.round().astype(int).clip(0, 60)
ant = ant.merge(lms, on=['sexo', 'mes'], how='left', validate='m:1')
ant['z_endi'] = rd.puntaje_z(ant.medida, ant.L, ant.M, ant.S)
n_z = int(ant.z_endi.notna().sum())
ant['z_endi'] = ant.z_endi.where(ant.z_endi.between(-rd.Z_MAX, rd.Z_MAX))

print(f'puntaje Z de la encuesta: {n_z:,} calculables, {int(ant.z_endi.notna().sum()):,} con |Z| <= 6')
print(f'proporción con Z < -2 (0 a 59 meses): {(ant.z_endi < -2).mean():.1%}')

## 5. Calidad del registro del carné

Se compara la última medición válida del carné con la medición estandarizada de la encuesta, cuando la primera
se tomó entre tres meses antes y un mes después de la segunda. La diferencia (carné − encuesta) resume el sesgo
sistemático (media) y el error aleatorio (desviación estándar).

In [ ]:
# última medición válida del carné de cada niño
ult = (L2.dropna(subset=['z_ok']).sort_values('edad')
         .groupby(K).tail(1)[K + ['edad', 'z_ok']]
         .rename(columns={'edad': 'edad_carne', 'z_ok': 'z_carne'}))

comp = ant[K + ['z_endi', 'edad_m']].merge(ult, on=K, how='inner')
comp['dif_edad'] = comp.edad_m - comp.edad_carne          # meses entre la medición del carné y la de la encuesta
comp = comp[comp.dif_edad.between(-1, 3)].dropna(subset=['z_endi', 'z_carne']).copy()
comp['dif'] = comp.z_carne - comp.z_endi

con_ambas = ant.loc[ant.z_endi.notna(), K + ['edad_m']].merge(ult, on=K, how='inner')
FLUJO.update(n_a_ambas=len(con_ambas),
             n_a_excl=int((~(con_ambas.edad_m - con_ambas.edad_carne).between(-1, 3)).sum()),
             n_calibracion=len(comp))

sesgo_global = comp.dif.mean()
de_dif = comp.dif.std()
lim_inf, lim_sup = sesgo_global - 1.96 * de_dif, sesgo_global + 1.96 * de_dif
ref_dci = comp.z_endi < -2
s0, e0 = sens_espec(comp.z_carne < -2, ref_dci)
s1, e1 = sens_espec(comp.z_carne - sesgo_global < -2, ref_dci)

CIFRAS.update({
    'calidad.n_pares': len(comp), 'calidad.n_upm': int(comp.id_upm.nunique()),
    'calidad.sesgo_global': sesgo_global, 'calidad.de_diferencia': de_dif,
    'calidad.lim_inf': lim_inf, 'calidad.lim_sup': lim_sup,
    'calidad.fuera_limites': ((comp.dif < lim_inf) | (comp.dif > lim_sup)).mean(),
    'calidad.correlacion': comp.z_endi.corr(comp.z_carne),
    'calidad.sens_sin_corregir': s0, 'calidad.espec_sin_corregir': e0,
    'calidad.sens_corregida': s1, 'calidad.espec_corregida': e1,
})
print(f'pares carné-encuesta: {len(comp):,} ({comp.id_upm.nunique():,} UPM)')
print(f'sesgo sistemático: {sesgo_global:+.3f} DE | error aleatorio: {de_dif:.3f} DE | '
      f'límites de concordancia 95%: {lim_inf:+.2f} a {lim_sup:+.2f} DE')
print(f"fuera de los límites: {CIFRAS['calidad.fuera_limites']:.1%} | correlación: {CIFRAS['calidad.correlacion']:.3f}")
print(f'DCI según el carné, sin corregir: sensibilidad {s0:.1%}, especificidad {e0:.1%}')
print(f'DCI según el carné, corregido:    sensibilidad {s1:.1%}, especificidad {e1:.1%}')

### 5.1. ¿El sesgo es constante?

Tendencia con la edad de la medición del carné y con el nivel de talla. Para el nivel de talla se usa la
regresión de la diferencia sobre la medición de referencia, recomendada cuando la referencia es mucho más
precisa que el método evaluado (Krouwer, 2008); la regresión sobre el promedio de ambas (gráfico de
Bland-Altman) se muestra como comparación.

In [ ]:
comp['tramo_edad'] = pd.cut(comp.edad_carne, [0, 12, 24, 36, 48, 60],
                            labels=['0-11', '12-23', '24-35', '36-47', '48-59'], right=False)
por_edad = comp.groupby('tramo_edad', observed=True).dif.agg(['mean', 'std', 'count'])
print('diferencia media por edad de la medición del carné (meses):')
print(por_edad.round(3).to_string())
tend_edad = stats.linregress(comp.edad_carne, comp.dif)
print(f'pendiente: {tend_edad.slope:+.4f} DE por mes (p = {tend_edad.pvalue:.1e})')

comp['dci_ref'] = comp.z_endi < -2
por_talla = comp.groupby('dci_ref').dif.agg(['mean', 'std', 'count'])
t_talla, p_talla = stats.ttest_ind(comp.loc[comp.dci_ref, 'dif'], comp.loc[~comp.dci_ref, 'dif'], equal_var=False)
print('\ndiferencia media según DCI en la medición de referencia:')
print(por_talla.round(3).to_string())
print(f't = {t_talla:.2f} (p = {p_talla:.1e})')

krouwer = stats.linregress(comp.z_endi, comp.dif)
promedio = (comp.z_carne + comp.z_endi) / 2
bland_altman = stats.linregress(promedio, comp.dif)
print(f'\nregresión de la diferencia sobre la referencia: pendiente {krouwer.slope:+.3f} '
      f'(EE {krouwer.stderr:.3f}; p = {krouwer.pvalue:.1e})')
print(f'regresión de la diferencia sobre el promedio:   pendiente {bland_altman.slope:+.3f} '
      f'(EE {bland_altman.stderr:.3f}; p = {bland_altman.pvalue:.1e})')

CIFRAS.update({f'calidad.sesgo_edad_{t.replace("-", "_")}': por_edad.loc[t, 'mean'] for t in por_edad.index})
CIFRAS.update({f'calidad.de_edad_{t.replace("-", "_")}': por_edad.loc[t, 'std'] for t in por_edad.index})
CIFRAS.update({f'calidad.n_edad_{t.replace("-", "_")}': int(por_edad.loc[t, 'count']) for t in por_edad.index})
CIFRAS.update({'calidad.pendiente_edad': tend_edad.slope, 'calidad.p_pendiente_edad': tend_edad.pvalue,
               'calidad.sesgo_con_dci': por_talla.loc[True, 'mean'], 'calidad.sesgo_sin_dci': por_talla.loc[False, 'mean'],
               'calidad.t_nivel_talla': t_talla, 'calidad.p_nivel_talla': p_talla,
               'calidad.pendiente_krouwer': krouwer.slope, 'calidad.p_krouwer': krouwer.pvalue,
               'calidad.pendiente_bland_altman': bland_altman.slope, 'calidad.p_bland_altman': bland_altman.pvalue})

### 5.2. Intervalo de tiempo entre ambas mediciones

El carné se midió hasta tres meses antes que la encuesta y, en la primera infancia, el puntaje *Z* tiende a
descender con la edad. Parte de la diferencia puede ser crecimiento real ocurrido entre ambas mediciones y no
error de registro; los pares separados por un mes o menos aíslan mejor el error del carné.

In [ ]:
comp_18 = comp[comp.edad_carne <= rd.EDAD_MAX_PREDICTORES]       # mediciones del carné hasta los 18 meses

for etiqueta, base in [('todas las edades', comp), ('carné hasta los 18 meses', comp_18)]:
    tramo = pd.cut(base.dif_edad, [-1.001, 0, 1, 2, 3.001], labels=['-1 a 0', '0 a 1', '1 a 2', '2 a 3'])
    print(f'=== {etiqueta} (n = {len(base):,}) ===')
    print(base.groupby(tramo, observed=True).dif.agg(['mean', 'std', 'count']).round(3).to_string())
    reg = mco(base.dif, [base.dif_edad, base.edad_carne], ['meses_entre_mediciones', 'edad_carne'])
    print(reg.round(4).to_string(), '\n')
    sufijo = 'todas' if base is comp else '18m'
    CIFRAS[f'calidad.coef_meses_{sufijo}'] = reg.loc['meses_entre_mediciones', 'coef']
    CIFRAS[f'calidad.p_coef_meses_{sufijo}'] = reg.loc['meses_entre_mediciones', 'p']

cerca = comp[comp.dif_edad.abs() <= 1]
cerca_18 = comp_18[comp_18.dif_edad.abs() <= 1]
sesgo_1m = cerca.dif.mean()
s_c, e_c = sens_espec(cerca.z_carne < -2, cerca.z_endi < -2)
s_k, e_k = sens_espec(comp.z_carne - sesgo_1m < -2, ref_dci)
CIFRAS.update({'calidad.n_1mes': len(cerca), 'calidad.sesgo_1mes': sesgo_1m, 'calidad.de_1mes': cerca.dif.std(),
               'calidad.sesgo_1mes_18m': cerca_18.dif.mean(), 'calidad.n_1mes_18m': len(cerca_18),
               'calidad.sens_1mes_sin_corregir': s_c, 'calidad.espec_1mes_sin_corregir': e_c,
               'calidad.sens_correccion_1mes': s_k, 'calidad.espec_correccion_1mes': e_k,
               'calidad.proporcion_registro': sesgo_1m / sesgo_global})
print(f'pares con un mes o menos entre mediciones: n = {len(cerca):,}; diferencia media {sesgo_1m:+.3f} DE '
      f'(hasta los 18 meses: {cerca_18.dif.mean():+.3f} DE); sensibilidad sin corregir {s_c:.1%}')
print(f'fracción de la diferencia atribuible al registro: {sesgo_1m / sesgo_global:.0%}')
print(f'corrigiendo con {sesgo_1m:.3f} DE: sensibilidad {s_k:.1%}, especificidad {e_k:.1%} (n = {len(comp):,})')

### 5.3. Sesgo usado para corregir las variables del modelo

El modelo solo usa mediciones tomadas hasta los 18 meses, así que se corrigen con el sesgo estimado en esos
pares, no con el sesgo global.

In [ ]:
SESGO_MODELO = comp_18.dif.mean()
FLUJO['n_sesgo_18m'] = len(comp_18)
CIFRAS.update({'calidad.sesgo_18m': SESGO_MODELO, 'calidad.n_18m': len(comp_18),
               'clasificacion.corte_equivalente': -2 + SESGO_MODELO})
print(f'sesgo en mediciones hasta los 18 meses: {SESGO_MODELO:+.3f} DE (n = {len(comp_18):,}); '
      f'sesgo global: {sesgo_global:+.3f} DE')

tabla_calidad = pd.DataFrame([
    ['Pares carné-encuesta', len(comp)],
    ['Sesgo sistemático (diferencia media, carné − encuesta), DE', round(sesgo_global, 3)],
    ['Error aleatorio (DE de la diferencia), DE', round(de_dif, 3)],
    ['Límite inferior de concordancia del 95%, DE', round(lim_inf, 2)],
    ['Límite superior de concordancia del 95%, DE', round(lim_sup, 2)],
    ['Correlación entre ambas mediciones', round(CIFRAS['calidad.correlacion'], 3)],
    ['Sensibilidad para DCI, sin corregir', round(s0, 3)],
    ['Especificidad para DCI, sin corregir', round(e0, 3)],
    ['Sensibilidad para DCI, tras corregir el sesgo', round(s1, 3)],
    ['Especificidad para DCI, tras corregir el sesgo', round(e1, 3)],
    [f'Sesgo en mediciones del carné hasta los 18 meses (n = {len(comp_18)}), DE', round(SESGO_MODELO, 3)],
    [f'Diferencia media en pares con un mes o menos entre mediciones (n = {len(cerca)}), DE', round(sesgo_1m, 3)],
], columns=['indicador', 'valor'], dtype=object)
guardar_tabla(tabla_calidad, 'tabla_calidad_carne')

## 6. Muestra de modelado y variables de trayectoria

Resultado: DCI (`dcronica2_5`, indicador oficial de la base) medida en el levantamiento estandarizado entre los
24 y 42 meses. Predictores: variables de trayectoria construidas con las mediciones del carné hasta los 18 meses,
corregidas por el sesgo, en niños con al menos tres mediciones. La función `rd.rasgos_trayectoria` es la misma
que usa el módulo de inferencia.

In [ ]:
blanco = nin[(nin.edad_m >= 24) & (nin.edad_m <= 42) & nin.dcronica2_5.notna()][
    K + ['dcronica2_5', 'edad_m', 'prov', 'etnia', 'area', 'educ_madre', 'tipo_nac', 'sexo']]
blanco['y'] = pd.to_numeric(blanco.dcronica2_5, errors='coerce')


def agregar_socio(df):
    df = df.copy()
    df['etnia_c'] = df.etnia.map({1: 'indigena', 2: 'afro', 3: 'montubia', 4: 'mestiza', 5: 'blanca'})
    df['area_c'] = df.area.map({1: 'urbano', 2: 'rural'})
    df['educ_c'] = df.educ_madre.map({1: 'basica', 2: 'media', 3: 'superior'})
    df['tipo_c'] = df.tipo_nac.fillna('sin_dato')
    return df


def construir_muestra(sesgo):
    '''Variables de trayectoria con las mediciones hasta los 18 meses corregidas por `sesgo`, unidas al resultado
    a los 24-42 meses. Devuelve la muestra de modelado (al menos 3 mediciones) y la tabla de todos los niños.'''
    Zs = L2.dropna(subset=['z_ok']).copy()
    Zs['z'] = Zs.z_ok - sesgo
    Zs = Zs[Zs.edad <= rd.EDAD_MAX_PREDICTORES].sort_values(K + ['edad'])
    Rs = (Zs.groupby(K)
            .apply(lambda g: pd.Series(rd.rasgos_trayectoria(g.edad.values, g.z.values)), include_groups=False)
            .reset_index())
    Ms = Rs.merge(blanco, on=K, how='inner')
    Ms = Ms[Ms.n_med >= rd.MIN_MEDICIONES].reset_index(drop=True)
    return agregar_socio(Ms), Rs


M, R = construir_muestra(SESGO_MODELO)
blanco_R = blanco.merge(R[K + ['n_med']], on=K, how='inner')
FLUJO.update(n_b_carne_18m=len(R), n_b_resultado=len(blanco), n_b_ventana=len(blanco_R),
             n_b_excl_ventana=FLUJO['n_validos'] - len(blanco_R),
             n_b_excl_menos3=int((blanco_R.n_med < rd.MIN_MEDICIONES).sum()),
             n_modelado=len(M), prev_modelado=M.y.mean())
CIFRAS['modelado.n_upm'] = int(M.id_upm.nunique())
print(f'niños con alguna medición válida hasta los 18 meses: {len(R):,}')
print(f'niños de 24 a 42 meses con resultado: {len(blanco):,}; con mediciones hasta los 18 meses: {len(blanco_R):,}')
print(f'muestra de modelado (al menos 3 mediciones): {len(M):,} niños, {M.id_upm.nunique():,} UPM; '
      f'prevalencia de DCI {M.y.mean():.1%}')

In [ ]:
# conteos del diagrama de flujo, superposición entre muestras y figura
ids_calibracion = set(map(tuple, comp[K].values))
ids_18 = set(map(tuple, comp_18[K].values))
ids_modelo = set(map(tuple, M[K].values))
solapados = ids_calibracion & ids_modelo
FLUJO.update(n_solapamiento=len(solapados), n_solapamiento_18m=len(ids_18 & ids_modelo))

# ¿cambia el sesgo si se excluye a los niños que están en ambas muestras?
es_solapado = comp[K].apply(tuple, axis=1).isin(solapados)
sesgo_sin_solapados = comp.loc[~es_solapado, 'dif'].mean()
CIFRAS.update({'calidad.sesgo_sin_solapamiento': sesgo_sin_solapados,
               'calidad.cambio_sesgo_sin_solapamiento': abs(sesgo_sin_solapados - sesgo_global),
               'calidad.edad_max_encuesta_18m': comp_18.edad_m.max(), 'modelado.edad_min': M.edad_m.min()})
CIFRAS.update({f'flujo.{k}': v for k, v in FLUJO.items()})

print(f"superposición: {len(solapados)} niños ({len(solapados) / len(M):.1%} de la muestra de modelado); "
      f"con la submuestra hasta los 18 meses: {FLUJO['n_solapamiento_18m']}")
print(f'sesgo global sin los niños superpuestos: {sesgo_sin_solapados:+.4f} DE '
      f'(cambio de {abs(sesgo_sin_solapados - sesgo_global):.4f} DE)')
print(f'edad en la encuesta: máximo {comp_18.edad_m.max():.1f} meses en la submuestra hasta los 18 meses; '
      f'mínimo {M.edad_m.min():.1f} meses en la muestra de modelado')
print()
print(pd.Series(FLUJO).to_string())
figuras.figura_flujo(FLUJO, RESULTADOS / 'figuras' / 'figura1_flujo.png')

In [ ]:
# características de la muestra de modelado
etiquetas = {
    'sexo': {'M': 'Masculino', 'F': 'Femenino'},
    'area': {1: 'Urbana', 2: 'Rural'},
    'etnia': {4: 'Mestiza', 1: 'Indígena', 2: 'Afroecuatoriana', 3: 'Montubia', 5: 'Blanca'},
    'educ_c': {'basica': 'Básica', 'media': 'Media', 'superior': 'Superior'},
    'tipo_c': {'termino_aeg': 'A término, AEG', 'termino_peg': 'A término, PEG', 'prematuro_aeg': 'Prematuro, AEG',
               'prematuro_peg': 'Prematuro, PEG', 'sin_dato': 'Sin dato'},
}
titulos = {'sexo': 'Sexo', 'area': 'Área de residencia', 'etnia': 'Etnia', 'educ_c': 'Instrucción de la madre',
           'tipo_c': 'Tipo de nacimiento'}
filas = []
for var, cats in etiquetas.items():
    prop = M[var].value_counts(normalize=True, dropna=False)
    for j, (codigo, nombre) in enumerate(cats.items()):
        valor = prop.get(codigo, 0.0)
        filas.append([titulos[var] if j == 0 else '', nombre, round(valor, 3)])
        CIFRAS[f'descriptiva.{var}_{codigo}'] = valor
filas.insert(2, ['Edad al resultado, media (DE)', 'Meses', f'{M.edad_m.mean():.1f} ({M.edad_m.std():.1f})'])
filas += [['Mediciones del carné hasta los 18 meses, media (DE)', 'Número', f'{M.n_med.mean():.1f} ({M.n_med.std():.1f})'],
          ['Prevalencia de DCI a los 24-42 meses', '', round(M.y.mean(), 3)]]
tabla_descriptiva = guardar_tabla(pd.DataFrame(filas, columns=['caracteristica', 'categoria', 'valor']),
                                  'tabla_descriptiva')

# datos faltantes y concordancia del indicador oficial con el puntaje Z recalculado
chk = M[K + ['y']].merge(ant[K + ['z_endi']], on=K, how='left', validate='1:1')
CIFRAS.update({'descriptiva.edad_media': M.edad_m.mean(), 'descriptiva.edad_de': M.edad_m.std(),
               'descriptiva.n_med_media': M.n_med.mean(), 'descriptiva.n_med_de': M.n_med.std(),
               'descriptiva.edad_primera_media': M.edad_pri.mean(), 'descriptiva.edad_ultima_media': M.edad_ult.mean(),
               'descriptiva.pend_faltante': int(M.pend.isna().sum()), 'descriptiva.pend_faltante_pct': M.pend.isna().mean(),
               'descriptiva.educ_faltante': int(M.educ_c.isna().sum()),
               'descriptiva.concordancia_indicador': ((chk.y == 1) == (chk.z_endi < -2)).mean()})
print(tabla_descriptiva.to_string(index=False))
print(f"\npendiente sin dato (serie de menos de 3 meses): {CIFRAS['descriptiva.pend_faltante']} "
      f"({CIFRAS['descriptiva.pend_faltante_pct']:.1%}); instrucción materna sin dato: {CIFRAS['descriptiva.educ_faltante']}")
print(f"primera medición a los {M.edad_pri.mean():.1f} meses y última a los {M.edad_ult.mean():.1f} meses, en promedio")
print(f"concordancia de dcronica2_5 con Z < -2 recalculado: {CIFRAS['descriptiva.concordancia_indicador']:.1%}")
print(pd.crosstab(chk.y, chk.z_endi < -2, margins=True).to_string())

In [ ]:
# prevalencias sin ponderar y ponderadas por el factor de expansión: ¿por qué la muestra de modelado tiene más DCI?
fx = nin[K + ['fexp', 'dcronica2_5']].copy()
fx['fexp'] = pd.to_numeric(fx.fexp, errors='coerce')
fx['yy'] = pd.to_numeric(fx.dcronica2_5, errors='coerce')


def prevalencia(df):
    x = df[K].merge(fx, on=K, how='left', validate='1:1')
    ok = x.yy.notna() & x.fexp.notna()
    return int(ok.sum()), x.yy[ok].mean(), np.average(x.yy[ok], weights=x.fexp[ok])


grupos_prev = {
    '24_59': ('24 a 59 meses, todos', nin[(nin.edad_m >= 24) & (nin.edad_m < 60) & nin.dcronica2_5.notna()]),
    '24_42': ('24 a 42 meses, todos', blanco),
    '24_42_carne': ('24 a 42 meses, con mediciones del carné hasta los 18 meses', blanco_R),
    'modelado': ('muestra de modelado (al menos 3 mediciones)', M),
}
filas = []
for clave, (nombre, df) in grupos_prev.items():
    n, sp, po = prevalencia(df)
    filas.append([nombre, n, round(sp, 3), round(po, 3)])
    CIFRAS.update({f'prevalencia.{clave}_n': n, f'prevalencia.{clave}_sin_ponderar': sp, f'prevalencia.{clave}_ponderada': po})
tabla_prev = guardar_tabla(pd.DataFrame(filas, columns=['grupo', 'n', 'sin_ponderar', 'ponderada']), 'tabla_prevalencias')
print(tabla_prev.to_string(index=False))

## 7. Validación cruzada: conjuntos de predictores y algoritmos

Conjuntos de predictores (regresión logística): A, solo sociodemográficos; B, una medición previa (puntaje *Z* y
edad de la última medición); C, trayectoria completa; D, trayectoria y sociodemográficos. Sobre el conjunto C se
comparan además Random Forest y Gradient Boosting. Todos usan la misma partición en cinco pliegues agrupada por
UPM, de modo que los niños de un mismo conglomerado nunca están a la vez en entrenamiento y validación.

In [ ]:
RASGOS = rd.RASGOS
SOCIO = ['etnia_c', 'area_c', 'educ_c', 'tipo_c']
CONJUNTOS = {'A': ([], SOCIO), 'B': (['z_ult', 'edad_ult'], []), 'C': (RASGOS, []), 'D': (RASGOS, SOCIO)}
ALGORITMOS = {
    'RF': Pipeline([('imp', SimpleImputer(strategy='median')),
                    ('clf', RandomForestClassifier(n_estimators=500, max_depth=5, min_samples_leaf=20,
                                                   random_state=SEMILLA_MODELOS))]),
    'GB': HistGradientBoostingClassifier(max_depth=3, max_iter=200, random_state=SEMILLA_MODELOS),
}
NOMBRES = {'A': 'A. Sociodemográfico (RL)', 'B': 'B. Una medición previa (RL)',
           'C': 'C. Trayectoria completa (RL, modelo final)', 'D': 'D. Trayectoria + sociodemográfico (RL)',
           'RF': 'C. Trayectoria completa (Random Forest)', 'GB': 'C. Trayectoria completa (Gradient Boosting)'}

cv = GroupKFold(N_PLIEGUES)
y = M.y.astype(int)
grupos = M.id_upm
pred, pliegues = {}, {}
for k, (num, cat) in CONJUNTOS.items():
    pp = pipeline_logistico(num, cat)
    pred[k] = cross_val_predict(pp, M, y, groups=grupos, cv=cv, method='predict_proba')[:, 1]
    pliegues[k] = cross_val_score(pp, M, y, groups=grupos, cv=cv, scoring='roc_auc')
for k, modelo in ALGORITMOS.items():
    pred[k] = cross_val_predict(modelo, M[RASGOS], y, groups=grupos, cv=cv, method='predict_proba')[:, 1]
    pliegues[k] = cross_val_score(modelo, M[RASGOS], y, groups=grupos, cv=cv, scoring='roc_auc')

for k in NOMBRES:
    CIFRAS.update({f'desempeno.auc_{k}': roc_auc_score(y, pred[k]),
                   f'desempeno.pliegues_media_{k}': pliegues[k].mean(), f'desempeno.pliegues_de_{k}': pliegues[k].std()})
    print(f"{NOMBRES[k]:<46} AUC {CIFRAS[f'desempeno.auc_{k}']:.4f} | pliegues {pliegues[k].mean():.4f} "
          f"± {pliegues[k].std():.4f}")

## 8. Sensibilidad a la corrección del sesgo

Se repite la construcción de las variables sin corrección y con el sesgo global. En la regresión logística con
variables estandarizadas, restar una constante a todos los puntajes *Z* no cambia las predicciones: el
desplazamiento desaparece al estandarizar. Solo cambia el indicador de alguna medición por debajo de −2 DE.

In [ ]:
ESCENARIOS = {'sin_correccion': 0.0, 'sesgo_global': sesgo_global, 'sesgo_18m': SESGO_MODELO}
pred_escenario = {}
for nombre, sesgo in ESCENARIOS.items():
    Ms = M if nombre == 'sesgo_18m' else construir_muestra(sesgo)[0]
    assert len(Ms) == len(M) and (Ms[K].values == M[K].values).all(), 'la muestra debe ser la misma en los tres escenarios'
    pe = {}
    for k in ['B', 'C', 'D']:
        pe[k] = pred[k] if Ms is M else cross_val_predict(pipeline_logistico(*CONJUNTOS[k]), Ms, Ms.y.astype(int),
                                                           groups=Ms.id_upm, cv=cv, method='predict_proba')[:, 1]
        CIFRAS[f'sensibilidad.auc_{k}_{nombre}'] = roc_auc_score(y, pe[k])
    CIFRAS[f'sensibilidad.brier_C_{nombre}'] = brier_score_loss(y, pe['C'])
    CIFRAS[f'sensibilidad.bajo_alguna_{nombre}'] = Ms.bajo_alguna.mean()
    pred_escenario[nombre] = pe
    print(f"{nombre:<15} sesgo {sesgo:+.4f} | AUC B {CIFRAS[f'sensibilidad.auc_B_{nombre}']:.4f} | "
          f"C {CIFRAS[f'sensibilidad.auc_C_{nombre}']:.4f} | D {CIFRAS[f'sensibilidad.auc_D_{nombre}']:.4f} | "
          f"Brier C {CIFRAS[f'sensibilidad.brier_C_{nombre}']:.4f} | alguna medición < -2: {Ms.bajo_alguna.mean():.1%}")
pred['C_sin_correccion'] = pred_escenario['sin_correccion']['C']

## 9. Intervalos de confianza por bootstrap de conglomerados

Se remuestrean UPM con reemplazo (2.000 réplicas) sobre las predicciones de validación cruzada, lo que respeta la
correlación entre niños de un mismo conglomerado. Las diferencias de AUC se calculan en las mismas réplicas.

In [ ]:
yv = y.values
codigos_upm, grupo_upm = np.unique(M.id_upm.values, return_inverse=True)
indices_upm = [np.flatnonzero(grupo_upm == j) for j in range(len(codigos_upm))]
MODELOS_IC = ['A', 'B', 'C', 'D', 'RF', 'GB', 'C_sin_correccion']

rng = np.random.default_rng(SEMILLA_BOOT)
boot = {k: np.empty(N_BOOT) for k in MODELOS_IC}
for b in range(N_BOOT):
    sel = rng.integers(0, len(codigos_upm), len(codigos_upm))
    ii = np.concatenate([indices_upm[j] for j in sel])
    for k in MODELOS_IC:
        boot[k][b] = roc_auc_score(yv[ii], pred[k][ii])

for k in MODELOS_IC:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    CIFRAS.update({f'desempeno.auc_{k}': roc_auc_score(yv, pred[k]), f'desempeno.ic_inf_{k}': lo, f'desempeno.ic_sup_{k}': hi})
    print(f'{k:<17} AUC {roc_auc_score(yv, pred[k]):.3f} (IC 95%: {lo:.3f} a {hi:.3f})')

print('\ndiferencias pareadas de AUC:')
for a, b_ in [('B', 'A'), ('C', 'B'), ('D', 'C'), ('C', 'RF'), ('C', 'GB'), ('C', 'C_sin_correccion')]:
    dif = roc_auc_score(yv, pred[a]) - roc_auc_score(yv, pred[b_])
    lo, hi = np.percentile(boot[a] - boot[b_], [2.5, 97.5])
    CIFRAS.update({f'desempeno.dif_{a}_{b_}': dif, f'desempeno.dif_ic_inf_{a}_{b_}': lo, f'desempeno.dif_ic_sup_{a}_{b_}': hi})
    print(f'{a} − {b_:<17} {dif:+.3f} (IC 95%: {lo:+.3f} a {hi:+.3f})')

tabla_desempeno = guardar_tabla(pd.DataFrame(
    [[NOMBRES[k], round(CIFRAS[f'desempeno.auc_{k}'], 3), round(CIFRAS[f'desempeno.ic_inf_{k}'], 3),
      round(CIFRAS[f'desempeno.ic_sup_{k}'], 3), round(pliegues[k].mean(), 3), round(pliegues[k].std(), 3)] for k in NOMBRES],
    columns=['conjunto_algoritmo', 'auc', 'ic95_inf', 'ic95_sup', 'media_pliegues', 'de_pliegues']), 'tabla_desempeno')

## 10. Calibración y clasificación del modelo final

Modelo final: regresión logística con el conjunto C. Calibración por deciles de riesgo predicho, pendiente e
intercepto de calibración (Steyerberg y Vergouwe, 2014) y puntaje de Brier. Clasificación con el umbral de 0,20,
que prioriza la sensibilidad, y con el que maximiza el índice de Youden.

In [ ]:
p_final = pred['C']
d = pd.DataFrame({'p': p_final, 'y': yv})
deciles = d.groupby(pd.qcut(d.p, 10, labels=False)).agg(pred=('p', 'mean'), obs=('y', 'mean'), n=('y', 'size'))
deciles['dif'] = (deciles.pred - deciles.obs).abs()
print(deciles.round(3).to_string())

lp = logit(np.clip(p_final, 1e-6, 1 - 1e-6))
pend_cal = LogisticRegression(C=1e10, max_iter=5000).fit(lp.reshape(-1, 1), yv).coef_[0, 0]
inter_cal = brentq(lambda a: (yv - expit(a + lp)).sum(), -5, 5)       # intercepto con pendiente fija en 1
brier = brier_score_loss(yv, p_final)
brier_ref = ((yv - yv.mean()) ** 2).mean()
guardar_tabla(deciles.reset_index().rename(columns={'p': 'decil'}).round(4), 'tabla_calibracion_deciles')

CIFRAS.update({'calibracion.pendiente': pend_cal, 'calibracion.intercepto': inter_cal,
               'calibracion.max_dif_deciles': deciles.dif.max(),
               'calibracion.pred_decil_max_dif': deciles.loc[deciles.dif.idxmax(), 'pred'],
               'calibracion.deciles_dif_menor_003': int((deciles.dif < 0.03).sum()),
               'calibracion.media_predicha': p_final.mean(), 'calibracion.media_observada': yv.mean(),
               'desempeno.brier_C': brier, 'desempeno.brier_referencia': brier_ref})
print(f'\npendiente de calibración {pend_cal:.3f} | intercepto {inter_cal:+.3f} | Brier {brier:.4f} '
      f'(referencia que asigna la prevalencia: {brier_ref:.4f})')
print(f'diferencia máxima entre deciles: {deciles.dif.max():.3f}; deciles con diferencia < 0,03: '
      f"{CIFRAS['calibracion.deciles_dif_menor_003']}")

op = operativas(yv, p_final >= UMBRAL)
fpr, tpr, umbrales = roc_curve(yv, p_final)
j = np.argmax(tpr - fpr)
CIFRAS.update({f'clasificacion.{m}': v for m, v in op.items()})
CIFRAS.update({'clasificacion.umbral': UMBRAL, 'clasificacion.youden_umbral': umbrales[j],
               'clasificacion.youden_sens': tpr[j], 'clasificacion.youden_espec': 1 - fpr[j]})
print(f"\numbral {UMBRAL}: marcados {op['marcados']:.1%} | sensibilidad {op['sens']:.1%} | especificidad {op['espec']:.1%} | "
      f"VPP {op['vpp']:.1%} | VP {op['vp']} FP {op['fp']} FN {op['fn']} VN {op['vn']}")
print(f'umbral de Youden {umbrales[j]:.3f}: sensibilidad {tpr[j]:.1%} | especificidad {1 - fpr[j]:.1%}')

### 10.1. Reglas de corte del carné frente al modelo

Lo que haría el personal de salud con el carné: marcar a un niño si su última medición (o alguna) está por debajo
de −2 DE, con o sin corrección del sesgo. El modelo se evalúa también con el umbral que marca la misma proporción
de niños que cada regla corregida, para comparar la detección con el mismo número de niños marcados.

In [ ]:
reglas = {
    'regla_1': ('Última medición Z < -2, sin corregir', (M.z_ult + SESGO_MODELO) < -2),
    'regla_2': ('Última medición Z < -2, corregida', M.z_ult < -2),
    'regla_3': ('Alguna medición Z < -2, sin corregir', (M.z_min + SESGO_MODELO) < -2),
    'regla_4': ('Alguna medición Z < -2, corregida', M.z_min < -2),
    'modelo_umbral': (f'Modelo, p >= {UMBRAL:.2f}', p_final >= UMBRAL),
}
for clave, regla in [('modelo_igual_2', 'regla_2'), ('modelo_igual_4', 'regla_4')]:
    fraccion = reglas[regla][1].mean()
    umbral_eq = np.quantile(p_final, 1 - fraccion)
    CIFRAS[f'reglas.{clave}_umbral'] = umbral_eq
    reglas[clave] = (f'Modelo, p >= {umbral_eq:.3f} (misma proporción que la {regla.replace("_", " ")})', p_final >= umbral_eq)

filas = []
for clave, (nombre, marca) in reglas.items():
    o = operativas(yv, marca)
    CIFRAS.update({f'reglas.{clave}_{m}': v for m, v in o.items()})
    filas.append([nombre, round(o['marcados'], 3), o['vp'], o['fp'], o['fn'], o['vn'],
                  round(o['sens'], 3), round(o['espec'], 3), round(o['vpp'], 3)])
tabla_reglas = guardar_tabla(pd.DataFrame(filas, columns=['regla', 'marcados', 'vp', 'fp', 'fn', 'vn', 'sens', 'espec', 'vpp']),
                             'tabla_reglas_clasificacion')
print(f'reglas aplicadas a las mediciones hasta los 18 meses; «corregida» = se restó {SESGO_MODELO:.3f} DE '
      f'(equivale a un corte de {-2 + SESGO_MODELO:.2f} DE sobre el valor registrado)\n')
print(tabla_reglas.to_string(index=False))

## 11. Datos exportados y figuras

Se exportan, sin identificadores, los pares carné-encuesta y las predicciones de validación cruzada (el
conglomerado se recodifica como un índice). Las figuras se generan a partir de esos archivos con `figuras.py`.

In [ ]:
comp[['z_carne', 'z_endi', 'edad_carne']].round(5).to_csv(RESULTADOS / 'datos' / 'datos_bland_altman.csv', index=False)

exportar = pd.DataFrame({'y': yv, 'conglomerado': grupo_upm})
for k in MODELOS_IC:
    exportar[f'p_{k}'] = np.round(pred[k], 6)
exportar['p_C_sesgo_global'] = np.round(pred_escenario['sesgo_global']['C'], 6)
exportar['z_ult'] = M.z_ult.round(5).values
exportar['z_min'] = M.z_min.round(5).values
exportar.to_csv(RESULTADOS / 'datos' / 'predicciones_cv.csv', index=False)

figuras.figura_bland_altman(RESULTADOS / 'datos' / 'datos_bland_altman.csv', RESULTADOS / 'figuras' / 'figura2_bland_altman.png')
figuras.figura_roc_calibracion(RESULTADOS / 'datos' / 'predicciones_cv.csv', RESULTADOS / 'figuras' / 'figura3_roc_calibracion.png')
print('datos:', sorted(p.name for p in (RESULTADOS / 'datos').iterdir()))
print('figuras:', sorted(p.name for p in (RESULTADOS / 'figuras').iterdir()))

## 12. Modelo final y módulo de inferencia

El modelo final se entrena con toda la muestra de modelado y se guarda como `modelo/parametros_modelo.json`
(medianas de imputación, medias y escalas de estandarización, coeficientes, sesgo del carné y tabla LMS). El
módulo `riesgo_dci.py` lo usa sin depender de scikit-learn; aquí se comprueba que ambos producen las mismas
probabilidades.

In [ ]:
modelo_final = pipeline_logistico(RASGOS).fit(M[RASGOS], y)
rd.exportar_parametros(
    modelo_final, lms, SESGO_MODELO, MODELO / 'parametros_modelo.json', umbral=UMBRAL,
    metadatos={'fuente': 'ENDI Ronda 2 (2023-2024), INEC', 'n_entrenamiento': len(M), 'prevalencia': round(M.y.mean(), 4),
               'auc_validacion_cruzada': round(CIFRAS['desempeno.auc_C'], 4), 'versiones': VERSIONES,
               'fecha': date.today().isoformat()})

modelo = rd.ModeloRiesgoDCI.cargar(MODELO / 'parametros_modelo.json')
p_modulo = np.array([modelo.probabilidad_desde_rasgos(f) for f in M[RASGOS].to_dict('records')])
p_sklearn = modelo_final.predict_proba(M[RASGOS])[:, 1]
assert np.abs(p_modulo - p_sklearn).max() < 1e-9, 'el módulo no reproduce las probabilidades del modelo'
print(f'módulo verificado: diferencia máxima con scikit-learn {np.abs(p_modulo - p_sklearn).max():.1e}')

coef = pd.DataFrame({'variable': RASGOS, 'coeficiente_estandarizado': modelo_final.named_steps['clf'].coef_[0],
                     'odds_ratio_por_de': np.exp(modelo_final.named_steps['clf'].coef_[0])})
guardar_tabla(coef.round(4), 'coeficientes_modelo_final')
print(coef.round(3).to_string(index=False))

# dos trayectorias de ejemplo (niño, cuatro mediciones entre los 2 y los 17 meses)
normal = [(2, 57.5), (6, 67.0), (12, 75.5), (17, 81.0)]
descendente = [(2, 54.0), (6, 62.0), (12, 68.0), (17, 71.5)]
CIFRAS['inferencia.ejemplo_normal'] = modelo.probabilidad(normal, 'M')
CIFRAS['inferencia.ejemplo_descendente'] = modelo.probabilidad(descendente, 'M')
print(f"\ntrayectoria cercana a la mediana: {CIFRAS['inferencia.ejemplo_normal']:.1%} | "
      f"trayectoria descendente: {CIFRAS['inferencia.ejemplo_descendente']:.1%}")

## 13. Cifras y verificación

Se guardan todas las cifras en `resultados/cifras.json` y se comparan con los valores de referencia publicados,
redondeados al número de decimales con que se informaron. Una diferencia en las métricas de validación cruzada
suele indicar otra versión de scikit-learn; una diferencia en los tamaños de muestra, otra versión de los
microdatos o de la tabla LMS.

In [ ]:
def a_json(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return None if np.isnan(v) else float(v)
    return v

CIFRAS['versiones'] = VERSIONES
(RESULTADOS / 'cifras.json').write_text(
    json.dumps({k: a_json(v) for k, v in sorted(CIFRAS.items())}, ensure_ascii=False, indent=1), encoding='utf-8')

# valores de referencia: (clave, valor informado, decimales)
REFERENCIA = [
    # muestras (diagrama de flujo)
    ('flujo.n_ninos', 22331, 0), ('flujo.n_carne', 12495, 0), ('flujo.med_carne', 71403, 0),
    ('flujo.exc_rango', 503, 0), ('flujo.exc_lms', 969, 0), ('flujo.exc_z6', 177, 0),
    ('flujo.med_validas', 69754, 0), ('flujo.n_validos', 12283, 0), ('flujo.n_a_ambas', 12283, 0),
    ('flujo.n_a_excl', 7522, 0), ('flujo.n_calibracion', 4761, 0), ('flujo.n_b_ventana', 3074, 0),
    ('flujo.n_b_excl_ventana', 9209, 0), ('flujo.n_b_excl_menos3', 1017, 0), ('flujo.n_modelado', 2057, 0),
    ('flujo.prev_modelado', 0.226, 3), ('flujo.n_solapamiento', 364, 0), ('flujo.n_solapamiento_18m', 0, 0),
    ('flujo.n_sesgo_18m', 3584, 0), ('modelado.n_upm', 1407, 0),
    # características de la muestra
    ('descriptiva.sexo_M', 0.509, 3), ('descriptiva.edad_media', 31.9, 1), ('descriptiva.edad_de', 5.2, 1),
    ('descriptiva.area_1', 0.523, 3), ('descriptiva.etnia_4', 0.817, 3), ('descriptiva.etnia_1', 0.124, 3),
    ('descriptiva.etnia_2', 0.031, 3), ('descriptiva.etnia_3', 0.020, 3), ('descriptiva.etnia_5', 0.007, 3),
    ('descriptiva.educ_c_basica', 0.273, 3), ('descriptiva.educ_c_media', 0.500, 3), ('descriptiva.educ_c_superior', 0.227, 3),
    ('descriptiva.tipo_c_termino_aeg', 0.763, 3), ('descriptiva.tipo_c_termino_peg', 0.076, 3),
    ('descriptiva.tipo_c_prematuro_aeg', 0.053, 3), ('descriptiva.tipo_c_prematuro_peg', 0.005, 3),
    ('descriptiva.tipo_c_sin_dato', 0.103, 3), ('descriptiva.n_med_media', 6.9, 1), ('descriptiva.n_med_de', 3.1, 1),
    ('descriptiva.edad_primera_media', 2.4, 1), ('descriptiva.edad_ultima_media', 14.5, 1),
    ('descriptiva.pend_faltante', 36, 0), ('descriptiva.concordancia_indicador', 0.986, 3),
    ('prevalencia.24_59_ponderada', 0.164, 3), ('prevalencia.24_42_ponderada', 0.182, 3),
    ('prevalencia.24_42_carne_ponderada', 0.205, 3), ('prevalencia.modelado_ponderada', 0.225, 3),
    # calidad del carné
    ('calidad.sesgo_global', 0.388, 3), ('calidad.de_diferencia', 0.755, 3), ('calidad.lim_inf', -1.09, 2),
    ('calidad.lim_sup', 1.87, 2), ('calidad.fuera_limites', 0.053, 3), ('calidad.correlacion', 0.799, 3),
    ('calidad.sens_sin_corregir', 0.553, 3), ('calidad.espec_sin_corregir', 0.971, 3),
    ('calidad.sens_corregida', 0.759, 3), ('calidad.espec_corregida', 0.900, 3),
    ('calidad.sesgo_edad_0_11', 0.495, 3), ('calidad.de_edad_0_11', 0.850, 3), ('calidad.n_edad_0_11', 2614, 0),
    ('calidad.sesgo_edad_48_59', 0.092, 3), ('calidad.de_edad_48_59', 0.334, 3), ('calidad.n_edad_48_59', 80, 0),
    ('calidad.pendiente_edad', -0.012, 3), ('calidad.sesgo_con_dci', 0.516, 3), ('calidad.sesgo_sin_dci', 0.356, 3),
    ('calidad.t_nivel_talla', 6.30, 2), ('calidad.pendiente_krouwer', -0.112, 3), ('calidad.pendiente_bland_altman', 0.118, 3),
    ('calidad.coef_meses_todas', 0.14, 2), ('calidad.coef_meses_18m', 0.18, 2),
    ('calidad.n_1mes', 1765, 0), ('calidad.sesgo_1mes', 0.317, 3), ('calidad.sesgo_1mes_18m', 0.353, 3),
    ('calidad.sens_1mes_sin_corregir', 0.585, 3), ('calidad.sens_correccion_1mes', 0.718, 3),
    ('calidad.espec_correccion_1mes', 0.922, 3), ('calidad.proporcion_registro', 0.8, 1),
    ('calidad.sesgo_18m', 0.450, 3), ('calidad.cambio_sesgo_sin_solapamiento', 0.014, 3),
    ('calidad.edad_max_encuesta_18m', 21, 0), ('modelado.edad_min', 24, 0),
    # desempeño
    ('desempeno.auc_A', 0.641, 3), ('desempeno.ic_inf_A', 0.610, 3), ('desempeno.ic_sup_A', 0.671, 3),
    ('desempeno.auc_B', 0.867, 3), ('desempeno.ic_inf_B', 0.848, 3), ('desempeno.ic_sup_B', 0.886, 3),
    ('desempeno.auc_C', 0.876, 3), ('desempeno.ic_inf_C', 0.857, 3), ('desempeno.ic_sup_C', 0.893, 3),
    ('desempeno.auc_D', 0.875, 3), ('desempeno.ic_inf_D', 0.856, 3), ('desempeno.ic_sup_D', 0.892, 3),
    ('desempeno.auc_RF', 0.875, 3), ('desempeno.ic_inf_RF', 0.858, 3), ('desempeno.ic_sup_RF', 0.893, 3),
    ('desempeno.auc_GB', 0.859, 3), ('desempeno.ic_inf_GB', 0.838, 3), ('desempeno.ic_sup_GB', 0.878, 3),
    ('desempeno.auc_C_sin_correccion', 0.875, 3), ('desempeno.ic_inf_C_sin_correccion', 0.856, 3),
    ('desempeno.ic_sup_C_sin_correccion', 0.893, 3),
    ('desempeno.pliegues_media_A', 0.646, 3), ('desempeno.pliegues_de_A', 0.034, 3),
    ('desempeno.pliegues_media_B', 0.868, 3), ('desempeno.pliegues_de_B', 0.019, 3),
    ('desempeno.pliegues_media_C', 0.877, 3), ('desempeno.pliegues_de_C', 0.016, 3),
    ('desempeno.pliegues_media_D', 0.876, 3), ('desempeno.pliegues_de_D', 0.016, 3),
    ('desempeno.pliegues_media_RF', 0.878, 3), ('desempeno.pliegues_de_RF', 0.013, 3),
    ('desempeno.pliegues_media_GB', 0.860, 3), ('desempeno.pliegues_de_GB', 0.019, 3),
    ('desempeno.dif_B_A', 0.226, 3), ('desempeno.dif_ic_inf_B_A', 0.194, 3), ('desempeno.dif_ic_sup_B_A', 0.259, 3),
    ('desempeno.dif_C_B', 0.009, 3), ('desempeno.dif_ic_inf_C_B', 0.002, 3), ('desempeno.dif_ic_sup_C_B', 0.016, 3),
    ('desempeno.dif_D_C', -0.001, 3), ('desempeno.dif_ic_inf_D_C', -0.004, 3), ('desempeno.dif_ic_sup_D_C', 0.002, 3),
    ('desempeno.dif_C_RF', 0.000, 3), ('desempeno.dif_ic_inf_C_RF', -0.005, 3), ('desempeno.dif_ic_sup_C_RF', 0.006, 3),
    ('desempeno.dif_C_GB', 0.017, 3), ('desempeno.dif_ic_inf_C_GB', 0.007, 3), ('desempeno.dif_ic_sup_C_GB', 0.027, 3),
    ('desempeno.dif_C_C_sin_correccion', 0.001, 3), ('desempeno.dif_ic_inf_C_C_sin_correccion', -0.002, 3),
    ('desempeno.dif_ic_sup_C_C_sin_correccion', 0.003, 3),
    ('desempeno.brier_C', 0.110, 3), ('desempeno.brier_referencia', 0.175, 3),
    ('sensibilidad.brier_C_sin_correccion', 0.110, 3), ('sensibilidad.auc_C_sesgo_global', 0.876, 3),
    ('sensibilidad.auc_B_sin_correccion', 0.867, 3), ('sensibilidad.auc_B_sesgo_global', 0.867, 3),
    # calibración y clasificación
    ('calibracion.pendiente', 0.971, 3), ('calibracion.intercepto', 0.008, 3), ('calibracion.max_dif_deciles', 0.035, 3),
    ('calibracion.pred_decil_max_dif', 0.225, 3), ('calibracion.deciles_dif_menor_003', 9, 0),
    ('clasificacion.youden_umbral', 0.25, 2), ('clasificacion.youden_sens', 0.798, 3), ('clasificacion.youden_espec', 0.805, 3),
    ('clasificacion.corte_equivalente', -1.55, 2),
    ('reglas.modelo_igual_2_umbral', 0.315, 3), ('reglas.modelo_igual_4_umbral', 0.146, 3),
    # inferencia (ejemplos del módulo)
    ('inferencia.ejemplo_normal', 0.034, 3), ('inferencia.ejemplo_descendente', 0.951, 3),
]
# tabla de reglas de clasificación: marcados, VP, FP, FN, VN, sensibilidad, especificidad, VPP
TABLA_REGLAS = {
    'regla_1': (0.144, 226, 71, 239, 1521, 0.486, 0.955, 0.761),
    'regla_2': (0.284, 327, 258, 138, 1334, 0.703, 0.838, 0.559),
    'regla_3': (0.257, 304, 225, 161, 1367, 0.654, 0.859, 0.575),
    'regla_4': (0.426, 394, 483, 71, 1109, 0.847, 0.697, 0.449),
    'modelo_umbral': (0.367, 379, 375, 86, 1217, 0.815, 0.764, 0.503),
    'modelo_igual_2': (0.284, 343, 242, 122, 1350, 0.738, 0.848, 0.586),
    'modelo_igual_4': (0.426, 406, 471, 59, 1121, 0.873, 0.704, 0.463),
}
for regla, valores in TABLA_REGLAS.items():
    for metrica, valor in zip(['marcados', 'vp', 'fp', 'fn', 'vn', 'sens', 'espec', 'vpp'], valores):
        REFERENCIA.append((f'reglas.{regla}_{metrica}', valor, 0 if isinstance(valor, int) else 3))

filas = []
for clave, esperado, dec in REFERENCIA:
    obtenido = CIFRAS.get(clave)
    ok = obtenido is not None and abs(float(obtenido) - esperado) <= 0.5 * 10 ** -dec + 1e-9
    filas.append([clave, esperado, None if obtenido is None else round(float(obtenido), dec + 1), ok])
verificacion = pd.DataFrame(filas, columns=['cifra', 'referencia', 'obtenido', 'coincide'])
guardar_tabla(verificacion, 'verificacion_cifras')

n_ok = int(verificacion.coincide.sum())
print(f'{n_ok} de {len(verificacion)} cifras coinciden con los valores de referencia.')
if n_ok < len(verificacion):
    print('\nCifras que no coinciden:')
    print(verificacion[~verificacion.coincide].to_string(index=False))